# Embedding Quality Evaluator

Evaluates an **embedding model's retrieval quality** using relevance triplets.

Each triplet is `(query, relevant_document, irrelevant_document)`.
For each triplet the evaluator:
- Embeds all three texts using the model's `/v1/embeddings` endpoint
- Computes cosine similarity between `(query, relevant)` and `(query, irrelevant)`
- Checks whether the relevant document ranked higher (correct rank)

**Aggregate metrics:** Accuracy, Mean Similarity Gap, Mean Relevant/Irrelevant Similarity, Average Latency.

## 1. Imports

In [ ]:
from bellmira.evaluators import ModelEmbeddingQualityEvaluator

## 2. Prepare relevance triplets

Each triplet is `(query, relevant_doc, irrelevant_doc)`.
The examples below are multilingual to reflect real usage.

In [ ]:
TRIPLETS = [
    (
        "Como posso bloquear o meu cartão?",
        "Para bloquear o seu cartão de débito ou crédito, aceda à aplicação mobile e selecione 'Cartões' > 'Bloquear Cartão'.",
        "As transferências internacionais podem demorar até 3 dias úteis dependendo do banco destinatário.",
    ),
    (
        "What are the steps to open a savings account?",
        "To open a savings account, visit a branch with your ID and proof of address, or complete the process online via the bank app.",
        "Credit cards offer various rewards programmes including cashback and airline miles.",
    ),
    (
        "Wie kann ich mein Passwort zurücksetzen?",
        "Um Ihr Passwort zurückzusetzen, klicken Sie auf 'Passwort vergessen' auf der Anmeldeseite und folgen Sie den Anweisungen.",
        "Unsere Filialen sind montags bis freitags von 9 bis 17 Uhr geöffnet.",
    ),
    (
        "What is a RAG pipeline?",
        "A RAG pipeline retrieves relevant documents from a knowledge base and feeds them as context to a language model before generation.",
        "Neural networks are trained using backpropagation and gradient descent.",
    ),
    (
        "Como funciona o MBway?",
        "O MBway permite realizar pagamentos e transferências imediatas usando apenas o número de telemóvel associado à conta.",
        "Os seguros de vida garantem proteção financeira aos beneficiários em caso de falecimento do segurado.",
    ),
]

## 3. Initialise the evaluator

In [ ]:
# Use an embeddings endpoint (not a chat endpoint)
EMBEDDING_URL = "https://embeddingsqwen3-8-backend.dat-aip-millm.qa.mbcp.cloud/"

evaluator = ModelEmbeddingQualityEvaluator(
    url=EMBEDDING_URL,
    triplets=TRIPLETS,
)

## 4. Quick check — embed a single text

In [ ]:
from bellmira.llm_model.llm_model_client import ModelClient

client = ModelClient(base_url=EMBEDDING_URL)
model_name = client.get_model_name()
print(f"Embedding model: {model_name}")

req = client.build_embedding_request(input_text="Hello world", model_name=model_name)
resp = client.send_request(req)
embedding = resp.json()["data"][0]["embedding"]
print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

## 5. Run evaluation

In [ ]:
results = evaluator.evaluate()

print(f"\n{'Query':<45}  {'sim_rel':>8}  {'sim_irr':>8}  {'gap':>8}  {'correct':>7}")
print("-" * 80)
for r in results:
    if r.get("Sim_gap") is not None:
        mark = "✓" if r["Correct_rank"] else "✗"
        print(f"{r['Query'][:43]:<45}  {r['Sim_relevant']:>8.4f}  "
              f"{r['Sim_irrelevant']:>8.4f}  {r['Sim_gap']:>8.4f}  {mark:>7}")
    else:
        print(f"{r['Query'][:43]:<45}  ERROR: {r.get('Error')}")

## 6. Extract aggregate metrics

In [ ]:
metrics = evaluator.extract_threshold_metrics(results)

print(f"Triplets evaluated:     {metrics['Triplets_evaluated']}")
print(f"Accuracy:               {metrics['Accuracy']:.1%}")
print(f"Mean similarity gap:    {metrics['Mean_sim_gap']:.4f}")
print(f"Mean sim (relevant):    {metrics['Mean_sim_relevant']:.4f}")
print(f"Mean sim (irrelevant):  {metrics['Mean_sim_irrelevant']:.4f}")
print(f"Avg latency per call:   {metrics['Avg_latency']}s")

## 7. Compare two embedding models

In [ ]:
models = {
    "QA embedding": "https://embeddingsqwen3-8-backend.dat-aip-millm.qa.mbcp.cloud/",
    "IN embedding": "https://embeddingsqwen3-8-backend.dat-aip-millm.in.mbcp.cloud/",
}

for model_label, url in models.items():
    ev = ModelEmbeddingQualityEvaluator(url=url, triplets=TRIPLETS)
    res = ev.evaluate()
    m = ev.extract_threshold_metrics(res)
    print(f"\n{model_label}: accuracy={m['Accuracy']:.1%}  gap={m['Mean_sim_gap']:.4f}  latency={m['Avg_latency']}s")